In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Connect to DuckDB and load cleaned dataset
conn = duckdb.connect()
conn.execute("CREATE TABLE supply_chain AS SELECT * FROM read_csv_auto('dataco_cleaned.csv')")

print("Table created successfully!")
print(conn.execute("SELECT COUNT(*) as total_rows FROM supply_chain").fetchdf())

## Query 1 — Sales & Shipping Performance by Market
**Business question:** Which market generates the most revenue and how fast do we deliver there?

**Techniques:** `GROUP BY`, `COUNT`, `SUM`, `AVG`, `ORDER BY`

In [ ]:
# Sales and shipping performance by market
query = """
    SELECT 
        Market,
        COUNT(*) AS total_orders,
        ROUND(SUM(Sales), 2) AS total_sales,
        ROUND(AVG("Days for shipping (real)"), 2) AS avg_shipping_days
    FROM supply_chain
    GROUP BY Market
    ORDER BY total_sales DESC
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

**Insight:** Despite global geographical spread, average shipping time is nearly identical across all markets (~3.5 days), suggesting well-standardized logistics processes worldwide. Europe and LATAM are the top 2 markets generating over 50% of total revenue.

## Query 2 — Late Delivery Rate by Shipping Mode
**Business question:** Which shipping mode has the highest late delivery rate?

**Techniques:** `CASE WHEN`, `HAVING`, conditional aggregation

In [ ]:
# Late delivery rate by shipping mode
query = """
    SELECT 
        "Shipping Mode",
        COUNT(*) AS total_orders,
        SUM(CASE WHEN "Late_delivery_risk" = 1 THEN 1 ELSE 0 END) AS late_orders,
        ROUND(SUM(CASE WHEN "Late_delivery_risk" = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS late_rate_pct
    FROM supply_chain
    GROUP BY "Shipping Mode"
    HAVING COUNT(*) > 1000
    ORDER BY late_rate_pct DESC
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

**Insight:** Classic supply chain paradox — premium shipping modes (First Class 95%, Second Class 77%) show significantly higher late delivery rates than Standard Class (38%). Tighter delivery windows leave no buffer for disruptions. `HAVING COUNT(*) > 1000` ensures statistical reliability by filtering out groups with insufficient data.

## Query 3 — Orders Above Global Average Sales (Subquery)
**Business question:** Which orders significantly exceed the global average sales value and by how much?

**Techniques:** `Subquery`, `WHERE`, `LIMIT`

In [ ]:
# Orders above average sales per market
query = """
    SELECT 
        Market,
        "Order Id",
        Sales,
        ROUND((SELECT AVG(Sales) FROM supply_chain), 2) AS avg_sales_global,
        ROUND(Sales - (SELECT AVG(Sales) FROM supply_chain), 2) AS above_avg_by
    FROM supply_chain
    WHERE Sales > (SELECT AVG(Sales) FROM supply_chain)
    ORDER BY Sales DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

**Insight:** The highest order value (~$2,000) is nearly 10x above the global average of $203. The concentration of top orders in Europe confirms it's the key market not just by volume but also by individual transaction value.

## Query 4 — Regional Late Delivery Analysis (CTE)
**Business question:** Which regions have the worst on-time delivery performance?

**Techniques:** `CTE (Common Table Expression)`, multi-step aggregation

In [ ]:
# Late delivery rate by region using CTE
query = """
    WITH regional_stats AS (
        SELECT 
            "Order Region",
            COUNT(*) AS total_orders,
            SUM(Late_delivery_risk) AS late_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct,
            ROUND(AVG("Days for shipping (real)"), 2) AS avg_shipping_days
        FROM supply_chain
        GROUP BY "Order Region"
    )
    SELECT *
    FROM regional_stats
    WHERE total_orders > 500
    ORDER BY late_rate_pct DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

**Insight:** Late deliveries are not a regional issue — they're a systemic problem. Since avg_shipping_days is identical across all regions (~3.5 days) but late rates exceed 55% everywhere, the root cause is likely unrealistic promised delivery windows rather than regional logistics inefficiency.

## Query 5 — Country Ranking Within Regions (Window Functions)
**Business question:** Which country within each region has the worst late delivery rate?

**Techniques:** `RANK() OVER (PARTITION BY)`, `AVG() OVER`, nested CTEs, statistical filtering

In [ ]:
# Regional performance ranking - filtered for statistical reliability
query = """
    WITH regional_stats AS (
        SELECT 
            "Order Region",
            "Order Country",
            COUNT(*) AS total_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct,
            ROUND(AVG(Sales), 2) AS avg_sales
        FROM supply_chain
        GROUP BY "Order Region", "Order Country"
        HAVING COUNT(*) > 200
    ),
    ranked AS (
        SELECT *,
            RANK() OVER (PARTITION BY "Order Region" ORDER BY late_rate_pct DESC) AS rank_in_region,
            ROUND(AVG(late_rate_pct) OVER (PARTITION BY "Order Region"), 2) AS avg_late_rate_region
        FROM regional_stats
    )
    SELECT *
    FROM ranked
    WHERE rank_in_region = 1
    ORDER BY late_rate_pct DESC
    LIMIT 10
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

**Insight:** Ecuador, Mozambique and Malaysia show the worst on-time performance per region. However, the consistent avg_late_rate_region across all regions (53-57%) confirms a systemic issue — no region stands out positively, suggesting the root cause is in global planning rather than local logistics.

> **Note on statistical reliability:** The first version of this query returned 100% late rate for countries with only 2-5 orders — statistically meaningless. Adding `HAVING COUNT(*) > 200` filters out small samples and ensures conclusions are based on sufficient data.

## Query 6 — Monthly Late Delivery Trend (CTE + Window Functions)
**Business question:** Is the late delivery problem improving or worsening over time?

**Techniques:** `DATE_TRUNC`, `CTE`, `AVG() OVER`, `ROWS BETWEEN 2 PRECEDING AND CURRENT ROW` (3-month rolling average)

In [ ]:
# Monthly late delivery trend with rolling average using CTE + Window Functions
query = """
    WITH monthly_stats AS (
        SELECT 
            DATE_TRUNC('month', "order date (DateOrders)") AS order_month,
            COUNT(*) AS total_orders,
            ROUND(SUM(Late_delivery_risk) * 100.0 / COUNT(*), 2) AS late_rate_pct
        FROM supply_chain
        GROUP BY DATE_TRUNC('month', "order date (DateOrders)")
    )
    SELECT 
        order_month,
        total_orders,
        late_rate_pct,
        ROUND(AVG(late_rate_pct) OVER (
            ORDER BY order_month
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 2) AS rolling_avg_3m
    FROM monthly_stats
    ORDER BY order_month
"""

result = conn.execute(query).fetchdf()
print(result.to_string())

**Insight:** Over the entire 2015-2017 period, the late delivery rate remains flat at 54-56% with no downward trend. The 3-month rolling average smooths out monthly anomalies and confirms this is not seasonal or one-off — it's a persistent structural problem requiring intervention at the delivery planning level, not operational fixes.

---

## Summary

| Technique | Query | Business Purpose |
|---|---|---|
| GROUP BY + HAVING | Query 1, 2 | Sales and late rate aggregation by market and shipping mode |
| CASE WHEN | Query 2 | Conditional counting of late orders |
| Subquery | Query 3 | Compare orders against global average |
| CTE | Query 4, 5, 6 | Multi-step analysis with readable intermediate results |
| RANK() OVER PARTITION BY | Query 5 | Country ranking within each region |
| AVG() OVER ROWS BETWEEN | Query 6 | 3-month rolling average to reveal trend |

**Key finding:** Late delivery is a systemic issue affecting all regions consistently at 54-56% over 3 years. Root cause: unrealistic promised delivery windows, not regional logistics inefficiency.